# CIGRE MT europeen — configuration radiale avec GFL

Reseau europeen de distribution moyenne tension de reference **CIGRE TF
C6.04.02**, alimente par un **reseau amont 110 kV** (bus infini, Ssc = 5000 MVA)
au travers de deux transformateurs 110/20 kV de 25 MVA. Exploitation
**radiale** : les trois interrupteurs de bouclage S1 (14-8), S2 (6-7) et
S3 (11-4) sont ouverts.

```
                 reseau amont 110 kV (bus 0, slack)
                    |                    |
                   T1                   T2
                    |                    |
  depart 1 : 1 - 2 - 3 - 4 - 5 - 6 (S2 ouvert)     depart 2 : 12 - 13 - 14
                     \                                             (S1 ouvert)
                      8 - 7 (S2)                    S3 : 11 - 4 ouvert
                      8 - 9 - 10 - 11
```

Trois convertisseurs **grid-following** sont raccordes : un aerogenerateur de
1,5 MVA au bus 7, deux PV de 1 MVA aux bus 9 et 13. Ils apportent chacun
8 etats (PLL, boucle de courant, boucles P/Q) et font apparaitre les modes de
synchronisation typiques du GFL en reseau faible.

Base systeme : **25 MVA, 50 Hz** (base = puissance d'un transfo HTB/HTA).

> Ce reseau n'a **pas** d'equivalent dans `matlab/G2Elin_V2/reseaux_test/` : il n'y a rien
> a comparer ligne a ligne pour l'instant. Le format des tableaux de donnees
> ci-dessous est celui attendu par `fromMatpower` / `buildFromMatrices` cote
> MATLAB, la transposition est donc directe.

### Comment comparer avec MATLAB

| Notebook (Python) | MATLAB |
|---|---|
| `net.run_loadflow()` | `net.Load_flow(Y_DER, false)` |
| `net.build_state_space()` | `net.buildGlobalModel()` |
| tableau modal ci-dessous | `displayModalAnalysis()` (`EigenVal_Table = 1`) |

Le tableau modal reprend **exactement** les conventions MATLAB : participation
`p_ki = |v_gauche . v_droit|` normalisee par mode, `f_undamp = |lambda|/2pi`,
`f_damp = |Im(lambda)|/2pi`, `zeta = -Re(lambda)/|lambda|`, tri par
`Re(lambda)` decroissant, tous les modes conjugues inclus. Le CSV exporte en
fin de notebook se compare ligne a ligne avec la table MATLAB.

> **Ecarts residuels.** Les equipements sont des portages fideles des classes
> MATLAB (memes equations, memes etats, memes valeurs par defaut) et les noeuds
> suivent la meme regle : capacite shunt quand les lignes en amenent, sinon
> resistance de causalite de 100 pu. La difference de fond qui reste est le
> **repere dq** : angle du slack cote MATLAB, repere synchrone commun ici, ce
> qui rend les modeles d'equipement locaux (aucun angle diffuse d'un element a
> l'autre). Les deux formulations se deduisent l'une de l'autre par une
> rotation globale : memes valeurs propres, angles decales d'une constante,
> mode de jauge (lambda = 0) porte par des etats differents. Voir
> [`g2egrid/docs/reperes_et_angles.md`](../docs/reperes_et_angles.md) et
> [`g2egrid/docs/equivalence_matlab.md`](../docs/equivalence_matlab.md).


In [9]:
import sys
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
import plotly.graph_objects as go


# Le paquet g2egrid est a la racine du projet (dossier parent de reseaux_test/).
ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from g2egrid import Network  # noqa: E402
from g2egrid.elements import (  # noqa: E402
    GFL1,
    GFL2,
    GFM1,
    GFM2,
    GFM3,
    GFM4,
    IB,
    Bus,
    ILoad,
    Line,
    PLoad,
    RLLoad,
    SynchronousMachine,
    SynchronousMachineTR,
    Transformer,
)

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

## 1. Construction du reseau

### 1.1 Donnees et assemblage

In [10]:
NOM_RESEAU = "CIGRE_MV"
F_ZOOM = 5.0

f0 = 50.0
Sb = 25e6

# BusType : 1 = slack, 2 = PV, 3 = PQ

Y_DER = [ # [BusID, BusType, V_base, V_pu, Theta_rad,   Pg_W, Qg_var,     Pl_W,   Ql_var]
        (    0,       1,      110e3, 1.030,        0,      0,      0,         0,       0),
        (    1,       3,       20e3, 1.000,        0,      0,      0,    20.4e6,     1e6),
        (    2,       3,       20e3, 1.000,        0,      0,      0,         0,       0),
        (    3,       3,       20e3, 1.000,        0,      0,      0,     550e3,    50e3),
        (    4,       3,       20e3, 1.000,        0,      0,      0,     445e3,    50e3),
        (    5,       3,       20e3, 1.000,        0,      0,      0,     750e3,    50e3),
        (    6,       3,       20e3, 1.000,        0,      0,      0,     565e3,    50e3),
        (    7,       3,       20e3, 1.000,        0,  1.2e6,      0,      90e3,    50e3),
        (    8,       3,       20e3, 1.000,        0,      0,      0,     605e3,    50e3),
        (    9,       3,       20e3, 1.000,        0,  0.8e6,      0,     675e3,    50e3),
        (   10,       3,       20e3, 1.000,        0,      0,      0,     570e3,    50e3),
        (   11,       3,       20e3, 1.000,        0,      0,      0,     340e3,    50e3),
        (   12,       3,       20e3, 1.000,        0,      0,      0,     205e3,    50e3),
        (   13,       3,       20e3, 1.000,        0,  0.8e6,      0,      40e3,    50e3),
        (   14,       3,       20e3, 1.000,        0,      0,      0,     605e3,    50e3),
]

T_Lines_SI = [
# T_Lines_SI : [ID, From, To, Length_km, R_ohm/km, X_ohm/km,    B_S/km]
              (  1,    1,  2,      2.82,    0.501,    0.716, 47.493e-6),
              (  2,    2,  3,      4.42,    0.501,    0.716, 47.493e-6),
              (  3,    3,  4,      0.61,    0.501,    0.716, 47.493e-6),
              (  4,    4,  5,      0.56,    0.501,    0.716, 47.493e-6),
              (  5,    5,  6,      1.54,    0.501,    0.716, 47.493e-6),
              (  6,    7,  8,      1.67,    0.501,    0.716, 47.493e-6),
              (  7,    8,  9,      0.32,    0.501,    0.716, 47.493e-6),
              (  8,    9, 10,      0.77,    0.501,    0.716, 47.493e-6),
              (  9,   10, 11,      0.33,    0.501,    0.716, 47.493e-6),
              ( 10,    3,  8,      1.30,    0.501,    0.716, 47.493e-6),
              ( 11,   12, 13,      4.89,    0.510,    0.366,  3.172e-6),
              ( 12,   13, 14,      2.99,    0.510,    0.366,  3.172e-6),
]

# T_Lines_pu : [ID, From, To, Length_km, R_pu/km, X_pu/km, B_pu/km, Un_V, Sb_VA]
T_Lines_pu = []

# T_Shunts : [ID, BusID, B_pu]   (compensation shunt propre au bus)
T_Shunts = []

#   110/20 kV, 25 MVA, uk = 12%, ukr = 0,16 % (base 25 MVA = Sb)
T_Transfos = [
 # T_Transfos : [ID, From, To, Sn_VA,  U1_V, U2_V,   R_pu,  X_pu]
               (  1,    0,  1,  25e6, 110e3, 20e3, 0.0016, 0.12),
               (  2,    0, 12,  25e6, 110e3, 20e3, 0.0016, 0.12),
]

#   reseau amont 110 kV : Ssc = 5000 MVA, R/X = 0,1
T_IBs = [
# T_IBs : [ID, BusID, Sn_VA, E_pu, theta_deg, R_pu, X_pu]
    (  1,  0, 5000e6, 1.030, 0, 0.0004975185951, 0.004975185951),
]

# T_SMs : [ID, BusID, Sn_VA]
T_SMs = []

# T_GFM1s a T_GFM4s : [ID, BusID, Sn_VA, Vdc_V]
T_GFM1s = []
T_GFM2s = []
T_GFM3s = []
T_GFM4s = []

# T_GFL1s : [ID, BusID, Sn_VA, Vdc_V]   (12 etats : LCL + PLL + P/Q + courant)
T_GFL1s = [
    (  1,  7, 1.5e6, 40e3),      # eolien
    (  2,  9, 1.0e6, 40e3),      # PV
    (  3, 13, 1.0e6, 40e3),      # PV
]
T_GFL2s = []
GFL_PARAMS = {}

# T_Loads : [ID, BusID, P_W, Q_var, Type]   (charges SPECIALES)
#   Type : 'I' = courant constant, 'P' = puissance constante. Les charges
#   classiques (R-L) viennent des colonnes Pl_W, Ql_var de Y_DER ; une charge
#   speciale s'y ajoute, load flow compris. Exemple : (1, 7, 0.5e6, 0.1e6, 'P'),
T_Loads = []

In [11]:
# --- bases de tension par bus, lues dans Y_DER ---
UN = {int(r[0]): r[2] for r in Y_DER}

# --- capacite shunt de chaque bus : moitie de la charge des lignes qui y
#     aboutissent (schema en pi), comme C_half = B_pu/2 dans Network.m. Un bus
#     qui n'en recoit aucune devient un noeud ALGEBRIQUE a resistance de
#     causalite R = 100 pu (Node.m / setResistor(100)). ---
Cbus = {int(r[0]): 0.0 for r in Y_DER}
for _id, i, j, km, _r, _x, b_km in T_Lines_SI:
    b_pu = b_km * km * (UN[i] ** 2 / Sb)
    Cbus[i] += b_pu / 2.0
    Cbus[j] += b_pu / 2.0
for _id, i, j, _km, _r, _x, b_pu, Un, Sb_l in T_Lines_pu:
    b_pu = b_pu * (UN[i] ** 2 / Sb) / (Un ** 2 / Sb_l)  # base de la ligne -> base systeme
    Cbus[i] += b_pu / 2.0
    Cbus[j] += b_pu / 2.0
# compensation shunt propre a un bus (banc de condensateurs), qui s'ajoute a
# la charge capacitive des lignes. MATLAB n'a pas de colonne pour cela dans
# Y_DER : d'ou une table a part, plutot qu'une colonne en plus.
for _id, b, B_pu in T_Shunts:
    Cbus[b] += B_pu

# --- charges speciales de T_Loads : leur consommation s'ajoute a celle de
#     Y_DER dans le load flow ---
CHARGES_SPECIALES = {"I": ILoad, "P": PLoad}
S_spec = {int(r[0]): [0.0, 0.0] for r in Y_DER}
for _id, b, P, Q, typ in T_Loads:
    if typ not in CHARGES_SPECIALES:
        raise ValueError(f"T_Loads, charge {_id} : type '{typ}' inconnu ('I' ou 'P')")
    S_spec[b][0] += P
    S_spec[b][1] += Q

# --- assemblage du reseau
net = Network(f0=f0, Sb=Sb)

for bid, btype, Un, Vpu, theta, Pg, Qg, Pl, Ql in Y_DER:  # Y_DER
    net.add(
        Bus(
            int(bid),
            Un,
            btype=int(btype),
            V=Vpu,
            theta=theta,
            Pg=Pg,
            Qg=Qg,
            Pl=Pl + S_spec[int(bid)][0],
            Ql=Ql + S_spec[int(bid)][1],
            C=Cbus[int(bid)],
        )
    )

for lid, i, j, km, r_km, x_km, _b in T_Lines_SI:  # T_Lines_SI
    ZB = UN[i] ** 2 / Sb
    net.add(
        Line(
            f"L{int(lid)}_{i}_{j}", i, j, R=r_km * km / ZB, X=x_km * km / ZB, wb=net.wb
        )
    )

for lid, i, j, _km, r_pu, x_pu, _b, Un, Sb_l in T_Lines_pu:  # T_Lines_pu
    k = (Un ** 2 / Sb_l) / (UN[i] ** 2 / Sb)  # base de la ligne -> base systeme
    net.add(Line(f"L{int(lid)}_{i}_{j}", i, j, R=r_pu * k, X=x_pu * k, wb=net.wb))

# Une prise hors nominal se declare par U1_V/U2_V, comme dans Network.m :
# n = (U1/U2)/(Un_i/Un_j), egal a 1 quand U1, U2 sont les tensions des bus.
for tid, i, j, Sn, U1, U2, R, X in T_Transfos:  # T_Transfos
    k = (Sb / Sn) * (U1 / UN[i]) ** 2  # base du transfo -> base systeme
    n = (U1 / U2) / (UN[i] / UN[j])
    net.add(Transformer(f"T{int(tid)}_{i}_{j}", i, j, R=R * k, X=X * k, wb=net.wb, n=n))

for _id, b, Sn, _E, _th, R, X in T_IBs:  # T_IBs
    net.add(IB(f"IB{b}", b, R=R, X=X, wb=net.wb, Sn=Sn, Sb=Sb))

# Colonnes optionnelles Rt_pu, Xt_pu des tables de DER : transfo de
# raccordement INTEGRE a l'equipement. Sans elles, la machine n'a pas de
# transfo (a declarer dans T_Transfos) et le convertisseur garde son filtre
# R2/L2 par defaut.
for _id, b, Sn, *Tr in T_SMs:  # T_SMs
    if Tr:
        net.add(SynchronousMachineTR(f"SM{b}", b, Sn=Sn, Sb=Sb, wb=net.wb, Rt=Tr[0], Lt=Tr[1], **SM_PARAMS))
    else:
        net.add(SynchronousMachine(f"SM{b}", b, Sn=Sn, Sb=Sb, wb=net.wb, **SM_PARAMS))

for T_GFM, GFM in ((T_GFM1s, GFM1), (T_GFM2s, GFM2), (T_GFM3s, GFM3), (T_GFM4s, GFM4)):  # T_GFM1s a T_GFM4s
    for _id, b, Sn, _Vdc, *Tr in T_GFM:
        tr = dict(R2_pu=Tr[0], L2_pu=Tr[1]) if Tr else {}
        net.add(GFM(f"GFM{b}", b, Sn=Sn, Sb=Sb, wb=net.wb, **(GFM_PARAMS | tr)))

for T_GFL, GFL in ((T_GFL1s, GFL1), (T_GFL2s, GFL2)):  # T_GFL1s, T_GFL2s
    for _id, b, Sn, _Vdc, *Tr in T_GFL:
        tr = dict(R2_pu=Tr[0], L2_pu=Tr[1]) if Tr else {}
        net.add(GFL(f"GFL{b}", b, Sn=Sn, Sb=Sb, wb=net.wb, **(GFL_PARAMS | tr)))

# Charges classiques : colonnes Pl_W, Ql_var de Y_DER -> charge R-L a
# impedance constante (GenericLoad.m). Une inductance ne peut pas consommer
# un reactif negatif ou nul : condensateur dans T_Shunts, ou charge speciale.
n_rl = 0
for bid, *_, Pl, Ql in Y_DER:  # Y_DER : charges classiques
    if Pl == 0 and Ql == 0:
        continue
    if Ql <= 0:
        raise ValueError(
            f"bus {int(bid)} : Ql = {Ql:g} var. Une charge R-L demande Ql > 0 ; "
            "declarer le condensateur dans T_Shunts ou la charge dans T_Loads ('P' ou 'I')"
        )
    net.add(RLLoad(f"Load{int(bid)}", int(bid), P=Pl / Sb, Q=Ql / Sb, wb=net.wb))
    n_rl += 1

for lid, b, P, Q, typ in T_Loads:  # T_Loads : charges speciales
    net.add(CHARGES_SPECIALES[typ](f"Load{typ}{int(lid)}_{b}", b, P=P / Sb, Q=Q / Sb))

n_gfm = len(T_GFM1s) + len(T_GFM2s) + len(T_GFM3s) + len(T_GFM4s)
print(
    f"{NOM_RESEAU} : {len(Y_DER)} bus, "
    f"{len(T_Lines_SI) + len(T_Lines_pu)} ligne(s), {len(T_Transfos)} transfo(s), "
    f"{len(T_IBs)} bus infini, {len(T_SMs)} machine(s), {n_gfm} GFM, "
    f"{len(T_GFL1s) + len(T_GFL2s)} GFL, {n_rl} charge(s) R-L, "
    f"{len(T_Loads)} charge(s) speciale(s)"
)

CIGRE_MV : 15 bus, 12 ligne(s), 2 transfo(s), 1 bus infini, 0 machine(s), 0 GFM, 3 GFL, 13 charge(s) R-L, 0 charge(s) speciale(s)


### 1.2 Load flow

In [12]:
lf = net.run_loadflow()
print(
    f"Load flow : converge en {lf.iterations} iterations (mismatch {lf.mismatch:.2e})\n"
)

print(
    f"{'bus':>4} {'|V| [pu]':>10} {'angle [deg]':>12} "
    f"{'P inj [pu]':>12} {'Q inj [pu]':>12}"
)
print("-" * 54)
for b, i in net._bus_index().items():
    print(
        f"{b:>4} {abs(lf.V[i]):>10.4f} {np.degrees(np.angle(lf.V[i])):>12.4f} "
        f"{lf.S[i].real:>12.4f} {lf.S[i].imag:>12.4f}"
    )
print(
    f"\nSomme des injections : P = {lf.S.real.sum():+.4f} pu, "
    f"Q = {lf.S.imag.sum():+.4f} pu  (= pertes)"
)

Load flow : converge en 5 iterations (mismatch 1.92e-14)

 bus   |V| [pu]  angle [deg]   P inj [pu]   Q inj [pu]
------------------------------------------------------
   0     1.0300       0.0000       0.9365       0.1560
   1     1.0167      -6.0631      -0.8160      -0.0400
   2     1.0060      -6.7544       0.0000       0.0000
   3     0.9890      -7.8461      -0.0220      -0.0020
   4     0.9875      -7.9541      -0.0178      -0.0020
   5     0.9865      -8.0289      -0.0300      -0.0020
   6     0.9853      -8.1166      -0.0226      -0.0020
   7     0.9903      -7.6685       0.0444      -0.0020
   8     0.9881      -7.8670      -0.0242      -0.0020
   9     0.9877      -7.8904       0.0050      -0.0020
  10     0.9867      -7.9593      -0.0228      -0.0020
  11     0.9865      -7.9699      -0.0136      -0.0020
  12     1.0293      -0.0129      -0.0082      -0.0020
  13     1.0299       0.0558       0.0304      -0.0020
  14     1.0275      -0.0239      -0.0242      -0.0020

Somme 

## 2. Modele d'etat

Equilibre element par element (Newton sur `[f; g] = 0`), puis assemblage
global : cablage des ports par LFT, elimination des variables algebriques
par complement de Schur.

In [13]:
net.solve_equilibrium()
ss = net.build_state_space()

print(ss)
print(f"\n{len(ss.states)} etats, {len(ss.inputs)} entrees, {len(ss.outputs)} sorties")
print(f"stable : {ss.is_stable()}   max Re(lambda) = {ss.eig().real.max():+.4e}")

print("\nEtats par element :")
seen = []
for s in ss.states:
    owner = s.split(".")[0]
    if owner not in seen:
        seen.append(owner)
for owner in seen:
    noms = [s.split(".", 1)[1] for s in ss.states if s.startswith(owner + ".")]
    print(f"  {owner:<12} {len(noms):>3} : {', '.join(noms)}")

<StateSpace 147 etats, 36 entrees, 119 sorties | max Re(lambda)=+0.000e+00>

147 etats, 36 entrees, 119 sorties
stable : True   max Re(lambda) = +0.0000e+00

Etats par element :
  bus1           2 : vd, vq
  bus2           2 : vd, vq
  bus3           2 : vd, vq
  bus4           2 : vd, vq
  bus5           2 : vd, vq
  bus6           2 : vd, vq
  bus7           2 : vd, vq
  bus8           2 : vd, vq
  bus9           2 : vd, vq
  bus10          2 : vd, vq
  bus11          2 : vd, vq
  bus12          2 : vd, vq
  bus13          2 : vd, vq
  bus14          2 : vd, vq
  L1_1_2         2 : i_d, i_q
  L2_2_3         2 : i_d, i_q
  L3_3_4         2 : i_d, i_q
  L4_4_5         2 : i_d, i_q
  L5_5_6         2 : i_d, i_q
  L6_7_8         2 : i_d, i_q
  L7_8_9         2 : i_d, i_q
  L8_9_10        2 : i_d, i_q
  L9_10_11       2 : i_d, i_q
  L10_3_8        2 : i_d, i_q
  L11_12_13      2 : i_d, i_q
  L12_13_14      2 : i_d, i_q
  T1_0_1         2 : i_d, i_q
  T2_0_12        2 : i_d, i_q
  IB0     

### 2.1 Tableau modal

Une ligne par valeur propre, triee par partie reelle decroissante (modes les
plus lents d'abord), comme `displayModalAnalysis` en MATLAB : une paire
conjuguee occupe deux lignes. L'index `Mode` est l'indice `k` de `lam[k]`.

| Colonne | Definition |
|---|---|
| `Re`, `Im` | parties reelle [1/s] et imaginaire [rad/s] de la valeur propre |
| `f_und [Hz]` | frequence propre non amortie, `abs(lambda) / 2pi` |
| `f_dam [Hz]` | frequence d'oscillation, `abs(Im) / 2pi` |
| `zeta [%]` | amortissement, `-Re / abs(lambda)` |
| `Etat j`, `%j` | j-ieme etat le plus participant et sa participation `abs(v_ki . w_ik)`, normalisee a 100 % par mode |

In [14]:
lam, Vr = np.linalg.eig(ss.A)
Wl = np.linalg.inv(Vr)
Pf = np.abs(Vr.T) * np.abs(Wl)  # Pf[k, i] : participation de l'etat i au mode k
Pf = Pf / np.maximum(Pf.sum(axis=1, keepdims=True), 1e-30)
top3 = np.argsort(-Pf, axis=1)[:, :3]  # les trois etats les plus participants
k = np.arange(len(lam))

mag = np.abs(lam)
zeta = np.where(mag > 1e-10, -lam.real / np.where(mag > 1e-10, mag, 1.0), 0.0)

T_eig = pd.DataFrame(
    {
        "Re": lam.real,
        "Im": lam.imag,
        "f_und [Hz]": mag / (2 * np.pi),
        "f_dam [Hz]": np.abs(lam.imag) / (2 * np.pi),
        "zeta [%]": 100 * zeta,
    }
)
for j in range(3):
    T_eig[f"Etat {j + 1}"] = [ss.states[i] for i in top3[:, j]]
    T_eig[f"%{j + 1}"] = 100 * Pf[k, top3[:, j]]
T_eig = T_eig.iloc[np.argsort(-lam.real, kind="stable")]
T_eig.index.name = "Mode"



T_eig

,Re,Im,f_und [Hz],f_dam [Hz],zeta [%],Etat 1,%1,Etat 2,%2,Etat 3,%3
Mode,,,,,,,,,,,
120,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,IB0.theta,100.000000,bus1.vd,0.000000,bus2.vd,0.000000
114,-2.944672e+01,0.000000,4.686591e+00,0.000000,100.000000,GFL7.M_q,35.014429,GFL9.M_q,32.616629,GFL7.M_d,14.453226
115,-2.974479e+01,0.000000,4.734031e+00,0.000000,100.000000,GFL9.M_q,29.950159,GFL7.M_q,29.596345,GFL9.M_d,20.911858
117,-3.002268e+01,0.000000,4.778258e+00,0.000000,100.000000,GFL9.M_d,40.458167,GFL9.M_q,26.024084,GFL7.M_d,18.941218
119,-3.093166e+01,0.000000,4.922926e+00,0.000000,100.000000,GFL13.M_q,64.458127,GFL13.M_d,33.821793,GFL13.M_pll,0.531531
...,...,...,...,...,...,...,...,...,...,...,...
73,-4.104284e+03,-314.160513,6.551279e+02,50.000199,99.708328,Load9.i_q,39.668576,Load9.i_d,39.666207,Load8.i_d,5.121886
70,-4.490727e+03,314.159637,7.164683e+02,50.000059,99.756193,Load5.i_q,42.980172,Load5.i_d,42.979450,Load9.i_d,3.057845
71,-4.490727e+03,-314.159637,7.164683e+02,50.000059,99.756193,Load5.i_q,42.980172,Load5.i_d,42.979450,Load9.i_d,3.057845


### 2.2 Carte des valeurs propres

In [15]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=T_eig["Re"], y=T_eig["Im"], mode="markers"))

fig.update_xaxes(range=[-1000, 10])
fig.update_yaxes(range=[-10, 1000])
fig.show()